In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path
from typing import Optional

# ---- COLAB SETUP (skip if running locally) ----
try:
    import google.colab as colab  # type: ignore
    _ = colab  # silence linters: imported for environment detection
    IN_COLAB = True
    print('Running in Google Colab')

    # Clone repo
    if not Path('/content/diffusion-lm-watermarking').exists():
        print('Cloning repo...')
        subprocess.check_call([
            'git', 'clone',
            'https://github.com/idhantsingh027/diffusion-lm-watermarking.git',
            '/content/diffusion-lm-watermarking',
        ])

    REPO_ROOT = Path('/content/diffusion-lm-watermarking')
    os.chdir(REPO_ROOT)

    # Install requirements
    print('Installing dependencies...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements.txt'),
    ])

    # Check GPU
    import torch
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Device: {device}')
    if device == 'cuda':
        print(f'GPU: {torch.cuda.get_device_name(0)}')
        print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
except ImportError:
    IN_COLAB = False
    print('Not in Colab, assuming local execution')

    def find_repo_root(start: Path) -> Path:
        for parent in [start] + list(start.parents):
            if (parent / 'requirements.txt').exists() and (parent / 'models').exists():
                return parent
        raise FileNotFoundError(
            'Could not locate repo root (requirements.txt + models/). '
            'Open the notebook from inside the repo.'
        )

    REPO_ROOT = find_repo_root(Path.cwd().resolve())
    os.chdir(REPO_ROOT)
    device = globals().get('device', 'cpu')

print('cwd (repo root):', Path.cwd())

# ---- HELPERS ----
def run_stream(cmd: list[str]) -> None:
    """Execute command and stream output in real-time."""
    cwd = str(REPO_ROOT)
    print('\n' + '=' * 80)
    print('Running:', ' '.join(map(str, cmd)))
    print('cwd:', cwd)
    print('=' * 80 + '\n')

    tail: list[str] = []
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        tail.append(line)
        if len(tail) > 80:
            tail.pop(0)
    rc = proc.wait()
    if rc != 0:
        print('\n--- last output lines (tail) ---')
        for line in tail:
            print(line, end='')
        raise RuntimeError(f'Command failed with exit code {rc}')

def latest_epoch_dir(root: Path) -> Optional[Path]:
    """Find the most recent epoch-N checkpoint in a directory."""
    if not root.exists():
        return None
    dirs = [p for p in root.iterdir() if p.is_dir() and p.name.startswith('epoch-')]
    if not dirs:
        return None
    def epoch_num(p: Path) -> int:
        try:
            return int(p.name.split('-')[-1])
        except Exception:
            return -1
    return sorted(dirs, key=lambda p: (epoch_num(p), p.name))[-1]

def prune_to_latest_epoch(root: Path) -> Optional[Path]:
    """Delete older epoch-* folders, keeping only the newest."""
    if not root.exists():
        return None
    dirs = [p for p in root.iterdir() if p.is_dir() and p.name.startswith('epoch-')]
    if not dirs:
        return None
    latest = latest_epoch_dir(root)
    assert latest is not None
    for d in dirs:
        if d != latest:
            shutil.rmtree(d, ignore_errors=True)
    return latest

# ---- TRAINING CONFIGURATION ----
OUTPUT_ROOT = (REPO_ROOT / 'checkpoints' / 'bert-mlm-curriculum-gradual').resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('output_root:', OUTPUT_ROOT)

# Toggles: set to False to skip specific stages
RUN_STAGE1 = True  # 15% masking (easy, BERT-like)
RUN_STAGE2 = True  # 30% masking (moderate)
RUN_STAGE3 = True  # 60% masking (hard)
RUN_STAGE4 = True  # 75% masking (very hard)
RUN_STAGE5 = True  # 90% masking (full diffusion)

# Training hyperparameters
BATCH_SIZE = 32
LR = 5e-5
MAX_LENGTH = 64
STEPS = 25
LOG_EVERY = 10

# Epoch budget: spend most time at 75–90% masking (total = 12 epochs, same as 4x3).
EPOCHS_15 = 1
EPOCHS_30 = 1
EPOCHS_60 = 2
EPOCHS_75 = 4
EPOCHS_90 = 4

# Each stage writes to its own folder so you keep distinct checkpoints.
STAGE1_DIR = OUTPUT_ROOT / 'stage1_maxp15'
STAGE2_DIR = OUTPUT_ROOT / 'stage2_maxp30'
STAGE3_DIR = OUTPUT_ROOT / 'stage3_maxp60'
STAGE4_DIR = OUTPUT_ROOT / 'stage4_maxp75'
STAGE5_DIR = OUTPUT_ROOT / 'stage5_maxp90'

def train_stage(*, stage_dir: Path, stage_label: str, model_name: str, max_mask_prob: float, epochs: int) -> Path:
    stage_dir.mkdir(parents=True, exist_ok=True)
    print('\n' + '-' * 80)
    print(f'{stage_label}: max_mask_prob={max_mask_prob} | epochs={epochs} | output_dir={stage_dir}')
    print(f'Resuming from: {model_name}')
    print('-' * 80 + '\n')
    cmd = [
        sys.executable, 'models/diffusion_lm.py', 'train',
        '--device', str(device),
        '--model_name', str(model_name),
        '--output_dir', str(stage_dir),
        '--epochs', str(int(epochs)),
        '--batch_size', str(BATCH_SIZE),
        '--lr', str(LR),
        '--max_length', str(MAX_LENGTH),
        '--steps', str(STEPS),
        '--min_mask_prob', '0.15',
        '--max_mask_prob', str(max_mask_prob),
        '--max_train_batches', '0',
        '--log_every', str(LOG_EVERY),
    ]
    run_stream(cmd)
    latest = prune_to_latest_epoch(stage_dir)
    if latest is None:
        raise RuntimeError(f'No checkpoint produced in {stage_dir}')
    print(f'Kept latest checkpoint: {latest}')
    return latest

# ---- STAGE 1 (15%) ----
stage1_ckpt = None
if RUN_STAGE1:
    stage1_ckpt = train_stage(
        stage_dir=STAGE1_DIR,
        stage_label='STAGE 1 (15%)',
        model_name='bert-base-uncased',
        max_mask_prob=0.15,
        epochs=EPOCHS_15,
    )
else:
    stage1_ckpt = latest_epoch_dir(STAGE1_DIR)
    print('Stage 1 skipped. Latest checkpoint:', stage1_ckpt)

# ---- STAGE 2 (30%) ----
stage2_ckpt = None
if RUN_STAGE2:
    stage2_model = str(stage1_ckpt) if stage1_ckpt is not None else 'bert-base-uncased'
    stage2_ckpt = train_stage(
        stage_dir=STAGE2_DIR,
        stage_label='STAGE 2 (30%)',
        model_name=stage2_model,
        max_mask_prob=0.30,
        epochs=EPOCHS_30,
    )
else:
    stage2_ckpt = latest_epoch_dir(STAGE2_DIR)
    print('Stage 2 skipped. Latest checkpoint:', stage2_ckpt)

# ---- STAGE 3 (60%) ----
stage3_ckpt = None
if RUN_STAGE3:
    stage3_model = str(stage2_ckpt) if stage2_ckpt is not None else 'bert-base-uncased'
    stage3_ckpt = train_stage(
        stage_dir=STAGE3_DIR,
        stage_label='STAGE 3 (60%)',
        model_name=stage3_model,
        max_mask_prob=0.60,
        epochs=EPOCHS_60,
    )
else:
    stage3_ckpt = latest_epoch_dir(STAGE3_DIR)
    print('Stage 3 skipped. Latest checkpoint:', stage3_ckpt)

# ---- STAGE 4 (75%) ----
stage4_ckpt = None
if RUN_STAGE4:
    stage4_model = str(stage3_ckpt) if stage3_ckpt is not None else 'bert-base-uncased'
    stage4_ckpt = train_stage(
        stage_dir=STAGE4_DIR,
        stage_label='STAGE 4 (75%)',
        model_name=stage4_model,
        max_mask_prob=0.75,
        epochs=EPOCHS_75,
    )
else:
    stage4_ckpt = latest_epoch_dir(STAGE4_DIR)
    print('Stage 4 skipped. Latest checkpoint:', stage4_ckpt)

# ---- STAGE 5 (90%) ----
stage5_ckpt = None
if RUN_STAGE5:
    stage5_model = str(stage4_ckpt) if stage4_ckpt is not None else 'bert-base-uncased'
    stage5_ckpt = train_stage(
        stage_dir=STAGE5_DIR,
        stage_label='STAGE 5 (90%)',
        model_name=stage5_model,
        max_mask_prob=0.90,
        epochs=EPOCHS_90,
    )
else:
    stage5_ckpt = latest_epoch_dir(STAGE5_DIR)
    print('Stage 5 skipped. Latest checkpoint:', stage5_ckpt)

print('\n' + '=' * 80)
print('Stage checkpoints:')
print(' - Stage 1 (15%):', stage1_ckpt)
print(' - Stage 2 (30%):', stage2_ckpt)
print(' - Stage 3 (60%):', stage3_ckpt)
print(' - Stage 4 (75%):', stage4_ckpt)
print(' - Stage 5 (90%):', stage5_ckpt)
print('=' * 80 + '\n')

# ---- ZIP FOR DOWNLOAD (Colab only) ----
if IN_COLAB:
    zip_path = '/content/bert-mlm-curriculum-gradual.zip'
    rel = OUTPUT_ROOT.relative_to(REPO_ROOT)
    print('Creating zip for download:', zip_path)
    subprocess.check_call(['zip', '-r', '-q', zip_path, str(rel)], cwd=str(REPO_ROOT))
    zip_size = Path(zip_path).stat().st_size / (1024**2)
    print(f'Zip created: {zip_path} ({zip_size:.1f} MB)')
    print('Download via:')
    print('  from google.colab import files')
    print(f'  files.download("{zip_path}")')

Running in Google Colab
Cloning repo...
Installing dependencies...
Device: cuda
GPU: Tesla T4
Memory: 15.8 GB
cwd (repo root): /content/diffusion-lm-watermarking
output_root: /content/diffusion-lm-watermarking/checkpoints/bert-mlm-curriculum-gradual

--------------------------------------------------------------------------------
STAGE 1 (15%): max_mask_prob=0.15 | epochs=1 | output_dir=/content/diffusion-lm-watermarking/checkpoints/bert-mlm-curriculum-gradual/stage1_maxp15
Resuming from: bert-base-uncased
--------------------------------------------------------------------------------


Running: /usr/bin/python3 models/diffusion_lm.py train --device cuda --model_name bert-base-uncased --output_dir /content/diffusion-lm-watermarking/checkpoints/bert-mlm-curriculum-gradual/stage1_maxp15 --epochs 1 --batch_size 32 --lr 5e-05 --max_length 64 --steps 25 --min_mask_prob 0.15 --max_mask_prob 0.15 --max_train_batches 0 --log_every 10
cwd: /content/diffusion-lm-watermarking


Generating test s

In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path
from typing import Optional

# ---- COLAB SETUP (skip if running locally) ----
try:
    import google.colab as colab  # type: ignore
    _ = colab  # silence linters: imported for environment detection
    IN_COLAB = True
    print('Running in Google Colab')

    # Clone repo
    if not Path('/content/diffusion-lm-watermarking').exists():
        print('Cloning repo...')
        subprocess.check_call([
            'git', 'clone',
            'https://github.com/idhantsingh027/diffusion-lm-watermarking.git',
            '/content/diffusion-lm-watermarking',
        ])

    REPO_ROOT = Path('/content/diffusion-lm-watermarking')
    os.chdir(REPO_ROOT)

    # Install requirements
    print('Installing dependencies...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements.txt'),
    ])

    # Check GPU
    import torch
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Device: {device}')
    if device == 'cuda':
        print(f'GPU: {torch.cuda.get_device_name(0)}')
        print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
except ImportError:
    IN_COLAB = False
    print('Not in Colab, assuming local execution')

    def find_repo_root(start: Path) -> Path:
        for parent in [start] + list(start.parents):
            if (parent / 'requirements.txt').exists() and (parent / 'models').exists():
                return parent
        raise FileNotFoundError(
            'Could not locate repo root (requirements.txt + models/). '
            'Open the notebook from inside the repo.'
        )

    REPO_ROOT = find_repo_root(Path.cwd().resolve())
    os.chdir(REPO_ROOT)
    device = globals().get('device', 'cpu')

print('cwd (repo root):', Path.cwd())

# ---- HELPERS ----
def run_stream(cmd: list[str]) -> None:
    """Execute command and stream output in real-time."""
    cwd = str(REPO_ROOT)
    print('\n' + '=' * 80)
    print('Running:', ' '.join(map(str, cmd)))
    print('cwd:', cwd)
    print('=' * 80 + '\n')

    tail: list[str] = []
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        tail.append(line)
        if len(tail) > 80:
            tail.pop(0)
    rc = proc.wait()
    if rc != 0:
        print('\n--- last output lines (tail) ---')
        for line in tail:
            print(line, end='')
        raise RuntimeError(f'Command failed with exit code {rc}')

def latest_epoch_dir(root: Path) -> Optional[Path]:
    """Find the most recent epoch-N checkpoint in a directory."""
    if not root.exists():
        return None
    dirs = [p for p in root.iterdir() if p.is_dir() and p.name.startswith('epoch-')]
    if not dirs:
        return None
    def epoch_num(p: Path) -> int:
        try:
            return int(p.name.split('-')[-1])
        except Exception:
            return -1
    return sorted(dirs, key=lambda p: (epoch_num(p), p.name))[-1]

def prune_to_latest_epoch(root: Path) -> Optional[Path]:
    """Delete older epoch-* folders, keeping only the newest."""
    if not root.exists():
        return None
    dirs = [p for p in root.iterdir() if p.is_dir() and p.name.startswith('epoch-')]
    if not dirs:
        return None
    latest = latest_epoch_dir(root)
    assert latest is not None
    for d in dirs:
        if d != latest:
            shutil.rmtree(d, ignore_errors=True)
    return latest

# ---- TRAINING CONFIGURATION ----
OUTPUT_ROOT = (REPO_ROOT / 'checkpoints' / 'bert-mlm-curriculum-gradual').resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('output_root:', OUTPUT_ROOT)

# Toggles: set to False to skip specific stages
RUN_STAGE1 = True  # 15% masking (easy, BERT-like)
RUN_STAGE2 = True  # 30% masking (moderate)
RUN_STAGE3 = True  # 60% masking (hard)
RUN_STAGE4 = True  # 75% masking (very hard)
RUN_STAGE5 = True  # 90% masking (full diffusion)

# Training hyperparameters
BATCH_SIZE = 32
LR = 5e-5
MAX_LENGTH = 64
STEPS = 25
LOG_EVERY = 10

# Epoch budget: spend most time at 75–90% masking (total = 12 epochs, same as 4x3).
EPOCHS_15 = 1
EPOCHS_30 = 1
EPOCHS_60 = 2
EPOCHS_75 = 4
EPOCHS_90 = 4

# Each stage writes to its own folder so you keep distinct checkpoints.
STAGE1_DIR = OUTPUT_ROOT / 'stage1_maxp15'
STAGE2_DIR = OUTPUT_ROOT / 'stage2_maxp30'
STAGE3_DIR = OUTPUT_ROOT / 'stage3_maxp60'
STAGE4_DIR = OUTPUT_ROOT / 'stage4_maxp75'
STAGE5_DIR = OUTPUT_ROOT / 'stage5_maxp90'

def train_stage(*, stage_dir: Path, stage_label: str, model_name: str, max_mask_prob: float, epochs: int) -> Path:
    stage_dir.mkdir(parents=True, exist_ok=True)
    print('\n' + '-' * 80)
    print(f'{stage_label}: max_mask_prob={max_mask_prob} | epochs={epochs} | output_dir={stage_dir}')
    print(f'Resuming from: {model_name}')
    print('-' * 80 + '\n')
    cmd = [
        sys.executable, 'models/diffusion_lm.py', 'train',
        '--device', str(device),
        '--model_name', str(model_name),
        '--output_dir', str(stage_dir),
        '--epochs', str(int(epochs)),
        '--batch_size', str(BATCH_SIZE),
        '--lr', str(LR),
        '--max_length', str(MAX_LENGTH),
        '--steps', str(STEPS),
        '--min_mask_prob', '0.15',
        '--max_mask_prob', str(max_mask_prob),
        '--max_train_batches', '0',
        '--log_every', str(LOG_EVERY),
    ]
    run_stream(cmd)
    latest = prune_to_latest_epoch(stage_dir)
    if latest is None:
        raise RuntimeError(f'No checkpoint produced in {stage_dir}')
    print(f'Kept latest checkpoint: {latest}')
    return latest

# ---- STAGE 1 (15%) ----
stage1_ckpt = None
if RUN_STAGE1:
    stage1_ckpt = train_stage(
        stage_dir=STAGE1_DIR,
        stage_label='STAGE 1 (15%)',
        model_name='bert-base-uncased',
        max_mask_prob=0.15,
        epochs=EPOCHS_15,
    )
else:
    stage1_ckpt = latest_epoch_dir(STAGE1_DIR)
    print('Stage 1 skipped. Latest checkpoint:', stage1_ckpt)

# ---- STAGE 2 (30%) ----
stage2_ckpt = None
if RUN_STAGE2:
    stage2_model = str(stage1_ckpt) if stage1_ckpt is not None else 'bert-base-uncased'
    stage2_ckpt = train_stage(
        stage_dir=STAGE2_DIR,
        stage_label='STAGE 2 (30%)',
        model_name=stage2_model,
        max_mask_prob=0.30,
        epochs=EPOCHS_30,
    )
else:
    stage2_ckpt = latest_epoch_dir(STAGE2_DIR)
    print('Stage 2 skipped. Latest checkpoint:', stage2_ckpt)

# ---- STAGE 3 (60%) ----
stage3_ckpt = None
if RUN_STAGE3:
    stage3_model = str(stage2_ckpt) if stage2_ckpt is not None else 'bert-base-uncased'
    stage3_ckpt = train_stage(
        stage_dir=STAGE3_DIR,
        stage_label='STAGE 3 (60%)',
        model_name=stage3_model,
        max_mask_prob=0.60,
        epochs=EPOCHS_60,
    )
else:
    stage3_ckpt = latest_epoch_dir(STAGE3_DIR)
    print('Stage 3 skipped. Latest checkpoint:', stage3_ckpt)

# ---- STAGE 4 (75%) ----
stage4_ckpt = None
if RUN_STAGE4:
    stage4_model = str(stage3_ckpt) if stage3_ckpt is not None else 'bert-base-uncased'
    stage4_ckpt = train_stage(
        stage_dir=STAGE4_DIR,
        stage_label='STAGE 4 (75%)',
        model_name=stage4_model,
        max_mask_prob=0.75,
        epochs=EPOCHS_75,
    )
else:
    stage4_ckpt = latest_epoch_dir(STAGE4_DIR)
    print('Stage 4 skipped. Latest checkpoint:', stage4_ckpt)

# ---- STAGE 5 (90%) ----
stage5_ckpt = None
if RUN_STAGE5:
    stage5_model = str(stage4_ckpt) if stage4_ckpt is not None else 'bert-base-uncased'
    stage5_ckpt = train_stage(
        stage_dir=STAGE5_DIR,
        stage_label='STAGE 5 (90%)',
        model_name=stage5_model,
        max_mask_prob=0.90,
        epochs=EPOCHS_90,
    )
else:
    stage5_ckpt = latest_epoch_dir(STAGE5_DIR)
    print('Stage 5 skipped. Latest checkpoint:', stage5_ckpt)

print('\n' + '=' * 80)
print('Stage checkpoints:')
print(' - Stage 1 (15%):', stage1_ckpt)
print(' - Stage 2 (30%):', stage2_ckpt)
print(' - Stage 3 (60%):', stage3_ckpt)
print(' - Stage 4 (75%):', stage4_ckpt)
print(' - Stage 5 (90%):', stage5_ckpt)
print('=' * 80 + '\n')

# ---- ZIP FOR DOWNLOAD (Colab only) ----
if IN_COLAB:
    # Use Python to build the zip (avoids relying on the system `zip` binary).
    rel = OUTPUT_ROOT.relative_to(REPO_ROOT)  # e.g. checkpoints/bert-mlm-curriculum-gradual
    base_name = '/content/bert-mlm-curriculum-gradual'  # make_archive will append .zip
    zip_path = base_name + '.zip'

    print('Creating zip for download:', zip_path)
    if Path(zip_path).exists():
        Path(zip_path).unlink()
    shutil.make_archive(
        base_name=base_name,
        format='zip',
        root_dir=str(REPO_ROOT),
        base_dir=str(rel),
    )

    zip_exists = Path(zip_path).exists()
    zip_size_mb = (Path(zip_path).stat().st_size / (1024**2)) if zip_exists else None
    print('zip exists:', zip_exists, '| size (MB):', zip_size_mb)
    print('Top-level /content listing:', sorted([p.name for p in Path('/content').iterdir()])[:50])

    # Persist to Google Drive (recommended for big files)
    print('Saving zip to Google Drive (so it survives runtime reset)...')
    from google.colab import drive  # type: ignore
    drive_root = Path('/content/drive')
    if not (drive_root / 'MyDrive').exists():
        drive.mount(str(drive_root))
    drive_path = drive_root / 'MyDrive' / 'bert-mlm-curriculum-gradual.zip'
    shutil.copy2(zip_path, drive_path)
    print('Saved to Drive:', drive_path)

    # Optional: try browser download too
    print('Download via:')
    print('  from google.colab import files')
    print(f'  files.download("{zip_path}")')
    try:
        from google.colab import files  # type: ignore
        files.download(zip_path)
    except Exception as e:
        print('Browser download failed (often due to size). Download from Drive instead.')
        print('Error:', repr(e))

Running in Google Colab
Cloning repo...
Installing dependencies...
Device: cuda
GPU: Tesla T4
Memory: 15.8 GB
cwd (repo root): /content/diffusion-lm-watermarking
output_root: /content/diffusion-lm-watermarking/checkpoints/bert-mlm-curriculum-gradual

--------------------------------------------------------------------------------
STAGE 1 (15%): max_mask_prob=0.15 | epochs=1 | output_dir=/content/diffusion-lm-watermarking/checkpoints/bert-mlm-curriculum-gradual/stage1_maxp15
Resuming from: bert-base-uncased
--------------------------------------------------------------------------------


Running: /usr/bin/python3 models/diffusion_lm.py train --device cuda --model_name bert-base-uncased --output_dir /content/diffusion-lm-watermarking/checkpoints/bert-mlm-curriculum-gradual/stage1_maxp15 --epochs 1 --batch_size 32 --lr 5e-05 --max_length 64 --steps 25 --min_mask_prob 0.15 --max_mask_prob 0.15 --max_train_batches 0 --log_every 10
cwd: /content/diffusion-lm-watermarking


Generating test s

ValueError: mount failed

In [1]:
# --- Test Basic Denoising ---

import os
import torch
from pathlib import Path
from transformers import BertTokenizerFast, BertForMaskedLM

class _TimestepConditionedBertForMaskedLM:
    """Minimal wrapper for timestep-conditioned BERT MLM."""
    @classmethod
    def from_pretrained(cls, ckpt_dir: str, num_steps: int, device: str):
        base = BertForMaskedLM.from_pretrained(ckpt_dir, local_files_only=True).to(device)
        time_embed_path = os.path.join(ckpt_dir, "time_embed.pt")
        if not os.path.exists(time_embed_path):
            raise FileNotFoundError(f"time_embed.pt not found at {time_embed_path}")
        time_embed_data = torch.load(time_embed_path, map_location=device)
        
        obj = cls()
        obj.bert = base
        obj.device = device
        obj.num_steps = time_embed_data["num_steps"]
        obj.time_embed = torch.nn.Embedding(obj.num_steps + 1, base.config.hidden_size).to(device)
        obj.time_embed.load_state_dict(time_embed_data["time_embed_state_dict"])
        obj.eval()
        return obj
    
    def eval(self):
        self.bert.eval()
        self.time_embed.eval()
        return self
    
    def __call__(self, input_ids, t, attention_mask=None):
        with torch.no_grad():
            if attention_mask is None:
                attention_mask = torch.ones_like(input_ids)
            
            # Get embeddings (must match training exactly!)
            tok_emb = self.bert.bert.embeddings.word_embeddings(input_ids)
            pos_emb = self.bert.bert.embeddings.position_embeddings(
                torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)
            )
            type_emb = self.bert.bert.embeddings.token_type_embeddings(torch.zeros_like(input_ids))
            
            # Add timestep embedding
            t_emb = self.time_embed(t).unsqueeze(1)
            emb = tok_emb + pos_emb + type_emb + t_emb
            
            # Forward through BERT
            outputs = self.bert.bert(
                inputs_embeds=emb,
                attention_mask=attention_mask,
                return_dict=True,
            )
            logits = self.bert.cls(outputs.last_hidden_state)
            return {"logits": logits}

def _get_excluded_token_ids(tokenizer):
    """Get set of token IDs to exclude from predictions (junk tokens)."""
    excluded_tokens = set()
    
    # Exclude specific problematic tokens
    problematic = ['@', '=', '#', '|', '{', '}', '[', ']', '##', '@@', '~', '^', '`']
    for tok in problematic:
        tok_id = tokenizer.convert_tokens_to_ids(tok)
        if tok_id != tokenizer.unk_token_id:
            excluded_tokens.add(tok_id)
    
    # Exclude tokens that are pure punctuation or special characters
    for token_id in range(len(tokenizer)):
        token = tokenizer.convert_ids_to_tokens(token_id)
        if token and all(c in '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~' for c in token):
            excluded_tokens.add(token_id)
        # Exclude tokens starting with ## that are pure punctuation
        if token and token.startswith('##') and all(c in '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~#' for c in token):
            excluded_tokens.add(token_id)
    
    return excluded_tokens

def _apply_vocab_mask_to_logits(logits, excluded_token_ids):
    """Mask excluded token IDs in logits tensor."""
    logits = logits.clone()
    excluded_list = list(excluded_token_ids)
    if len(excluded_list) > 0:
        logits[..., excluded_list] = -1e10
    return logits

def _mask_prob_for_t(t: torch.Tensor, steps: int, min_p: float, max_p: float) -> torch.Tensor:
    """Linear masking schedule."""
    ratio = (steps - t.float()) / steps
    return min_p + (max_p - min_p) * ratio

def _resolve_device(device: str) -> str:
    """Resolve device string."""
    if device == "auto":
        return "cuda" if torch.cuda.is_available() else "cpu"
    return device

def denoise_masked_sentence(
    ckpt_dir: str,
    masked_text: str,
    *,
    steps: int = 25,
    min_mask_prob: float = 0.15,
    max_mask_prob: float = 0.95,
    temperature: float = 1.0,
    top_k: int = 50,
    device: str = "auto",
) -> str:
    """
    Iteratively denoise a masked sentence using progressive unmasking.
    
    Args:
        ckpt_dir: Path to checkpoint directory
        masked_text: Text with [MASK] tokens
        steps: Number of diffusion steps (default: 25)
        min_mask_prob: Minimum masking probability (default: 0.15)
        max_mask_prob: Maximum masking probability (default: 0.95)
        temperature: Sampling temperature (default: 1.0)
        top_k: Top-k filtering (default: 50)
        device: "cpu", "cuda", or "auto"
    
    Returns:
        Denoised text with [MASK] tokens replaced
    """
    device = _resolve_device(device)
    tokenizer = BertTokenizerFast.from_pretrained(ckpt_dir, local_files_only=True)
    
    # Get excluded token IDs for vocab filtering
    excluded_token_ids = _get_excluded_token_ids(tokenizer)
    
    # Load model with timestep conditioning
    use_time = os.path.exists(os.path.join(ckpt_dir, "time_embed.pt"))
    if use_time:
        model = _TimestepConditionedBertForMaskedLM.from_pretrained(ckpt_dir, num_steps=steps, device=device)
    else:
        model = BertForMaskedLM.from_pretrained(ckpt_dir, local_files_only=True).to(device)
        model.eval()
    
    # Tokenize
    enc = tokenizer(masked_text, return_tensors="pt", add_special_tokens=True)
    input_ids = enc["input_ids"].to(device)
    attention_mask = enc.get("attention_mask", torch.ones_like(input_ids, dtype=torch.long)).to(device)
    
    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id
    pad_id = tokenizer.pad_token_id
    
    with torch.no_grad():
        for t_int in range(steps, 0, -1):
            t_tensor = torch.tensor([t_int], device=device)
            
            # Compute masking probabilities
            p_t = float(_mask_prob_for_t(t_tensor, steps=steps, min_p=min_mask_prob, max_p=max_mask_prob)[0])
            if t_int == 1:
                p_prev = 0.0
            else:
                p_prev = float(_mask_prob_for_t(torch.tensor([t_int - 1], device=device), steps=steps, min_p=min_mask_prob, max_p=max_mask_prob)[0])
            
            # Ensure no division by zero
            p_t = max(p_t, 1e-6)
            keep_mask_prob = min(max(p_prev / p_t, 0.0), 1.0)
            
            # Special tokens should never be masked
            special = input_ids.eq(cls_id) | input_ids.eq(sep_id)
            if pad_id is not None:
                special |= input_ids.eq(pad_id)
            
            is_mask = input_ids.eq(tokenizer.mask_token_id) & (~special)
            if not bool(is_mask.any()):
                break
            
            # Randomly keep some masks
            stay_masked = (torch.rand_like(input_ids.float()) < keep_mask_prob) & is_mask
            to_unmask = is_mask & (~stay_masked)
            
            if not bool(to_unmask.any()):
                continue
            
            # Get predictions
            if use_time:
                logits = model(input_ids=input_ids, t=t_tensor, attention_mask=attention_mask)["logits"]
            else:
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            
            # CRITICAL: Apply vocabulary mask to filter out junk tokens
            logits = _apply_vocab_mask_to_logits(logits, excluded_token_ids)
            
            # Temperature scaling
            logits = logits / max(float(temperature), 1e-6)
            
            # Top-k filtering
            if top_k > 0:
                topk_vals, topk_idx = torch.topk(logits, k=min(int(top_k), logits.shape[-1]), dim=-1)
                filtered = torch.full_like(logits, fill_value=-float("inf"))
                filtered.scatter_(-1, topk_idx, topk_vals)
                logits = filtered
            
            # Sample from filtered distribution
            probs = torch.softmax(logits, dim=-1)
            sampled = torch.multinomial(probs.view(-1, probs.size(-1)), num_samples=1).view(1, -1)
            
            # Replace masked tokens
            input_ids[to_unmask] = sampled[to_unmask]
    
    # Decode final result
    final_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    return " ".join(final_text.split())

# Get checkpoint path - UPDATED TO NEW MODEL
notebook_dir = Path.cwd()
repo_root = notebook_dir.parent if (notebook_dir / ".." / "requirements.txt").exists() else notebook_dir
CKPT = str((repo_root / "checkpoints" / "bert-mlm-curriculum-gradualv2" / "stage5").resolve())

# Test
test_sentence = "The quick brown [MASK] jumps over the lazy [MASK]."
print(f"Input:  {test_sentence}")
denoised = denoise_masked_sentence(CKPT, test_sentence, steps=25, device="auto")
print(f"Output: {denoised}")


C:\Users\Santosh Kumar Singh\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Input:  The quick brown [MASK] jumps over the lazy [MASK].
Output: the quick brown jumps over the lazy 7.


In [2]:
# --- Test denoising accuracy with known ground truth (PROGRESSIVE EVALUATION) ---

import re
from pathlib import Path

# Wikipedia-style test examples (matching training data distribution)
test_cases = [
    {
        "original": "The policy was announced during the annual conference in Washington.",
        "masked_words": ["policy", "conference"],
        "description": "Political/government context (Wikipedia-style)"
    },
    {
        "original": "The company was founded in 1995 by researchers from the university.",
        "masked_words": ["company", "university"],
        "description": "Historical/business context (Wikipedia-style)"
    },
    {
        "original": "The research was published in the journal by scientists from various countries.",
        "masked_words": ["research", "journal"],
        "description": "Academic/scientific context (Wikipedia-style)"
    },
    {
        "original": "The organization was established to promote cooperation between member states.",
        "masked_words": ["organization", "cooperation"],
        "description": "International relations (Wikipedia-style)"
    }
]

def create_masked_version(text: str, words_to_mask: list[str]) -> tuple[str, list[str]]:
    """Replace specified words with [MASK], return masked text and actual masked words."""
    masked_text = text
    actual_masked = []
    
    for word in words_to_mask:
        pattern = re.compile(r'\b' + re.escape(word) + r'\b', re.IGNORECASE)
        match = pattern.search(masked_text)
        if match:
            actual_masked.append(match.group())
            masked_text = pattern.sub('[MASK]', masked_text, count=1)
    
    return masked_text, actual_masked

def denoise_with_progressive_tracking(
    ckpt_dir: str,
    masked_text: str,
    *,
    steps: int = 25,
    min_mask_prob: float = 0.15,
    max_mask_prob: float = 0.75,  # REDUCED from 0.95
    temperature: float = 0.6,      # REDUCED from 1.0
    top_k: int = 100,              # INCREASED from 50
    device: str = "auto",
    track_every: int = 5,          # Track every N steps
) -> dict:
    """Denoise and track intermediate outputs at multiple diffusion steps."""
    import os
    import torch
    from transformers import BertTokenizerFast, BertForMaskedLM
    
    device = _resolve_device(device)
    tokenizer = BertTokenizerFast.from_pretrained(ckpt_dir, local_files_only=True)
    
    # Get excluded token IDs for vocab filtering
    excluded_token_ids = _get_excluded_token_ids(tokenizer)
    
    # Use timestep-conditioned model if available
    use_time = os.path.exists(os.path.join(ckpt_dir, "time_embed.pt"))
    if use_time:
        model = _TimestepConditionedBertForMaskedLM.from_pretrained(ckpt_dir, num_steps=steps, device=device)
    else:
        model = BertForMaskedLM.from_pretrained(ckpt_dir, local_files_only=True).to(device)
        model.eval()
    
    enc = tokenizer(masked_text, return_tensors="pt", add_special_tokens=True)
    input_ids = enc["input_ids"].to(device)
    attention_mask = enc.get("attention_mask", torch.ones_like(input_ids, dtype=torch.long)).to(device)
    
    cls_id, sep_id, pad_id = tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id
    
    # Track intermediate outputs
    intermediate_outputs = {}
    
    with torch.no_grad():
        for t_int in range(steps, 0, -1):
            t_tensor = torch.tensor([t_int], device=device)
            p_t = float(_mask_prob_for_t(t_tensor, steps=steps, min_p=min_mask_prob, max_p=max_mask_prob)[0])
            p_prev = 0.0 if t_int == 1 else float(_mask_prob_for_t(torch.tensor([t_int - 1], device=device), steps=steps, min_p=min_mask_prob, max_p=max_mask_prob)[0])
            
            p_t = max(p_t, 1e-6)
            keep_mask_prob = min(max(p_prev / p_t, 0.0), 1.0)
            
            special = input_ids.eq(cls_id) | input_ids.eq(sep_id)
            if pad_id is not None:
                special |= input_ids.eq(pad_id)
            
            is_mask = input_ids.eq(tokenizer.mask_token_id) & (~special)
            if not bool(is_mask.any()):
                break
            
            stay_masked = (torch.rand_like(input_ids.float()) < keep_mask_prob) & is_mask
            to_unmask = is_mask & (~stay_masked)
            if not bool(to_unmask.any()):
                continue
            
            if use_time:
                logits = model(input_ids=input_ids, t=t_tensor, attention_mask=attention_mask)["logits"]
            else:
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            
            # CRITICAL: Apply vocabulary mask to filter out junk tokens
            logits = _apply_vocab_mask_to_logits(logits, excluded_token_ids)
            
            logits = logits / max(float(temperature), 1e-6)
            if top_k > 0:
                topk_vals, topk_idx = torch.topk(logits, k=min(int(top_k), logits.shape[-1]), dim=-1)
                filtered = torch.full_like(logits, fill_value=-float("inf"))
                filtered.scatter_(-1, topk_idx, topk_vals)
                logits = filtered
            
            probs = torch.softmax(logits, dim=-1)
            sampled = torch.multinomial(probs.view(-1, probs.size(-1)), num_samples=1).view(1, -1)
            input_ids[to_unmask] = sampled[to_unmask]
            
            # Track this step if it's a milestone
            if t_int % track_every == 0 or t_int == 1:
                text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
                intermediate_outputs[t_int] = " ".join(text.split())
    
    final_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    return {
        "final": " ".join(final_text.split()),
        "intermediate": intermediate_outputs
    }

def evaluate_progressive_recovery(intermediate_outputs: dict, expected_words: list[str]) -> dict:
    """Check when each expected word first appears during denoising."""
    recovery_timeline = {}
    
    for word in expected_words:
        word_lower = word.lower()
        for step in sorted(intermediate_outputs.keys(), reverse=True):
            output = intermediate_outputs[step]
            if word_lower in output.lower():
                recovery_timeline[word] = step
                break
        if word not in recovery_timeline:
            recovery_timeline[word] = None  # Never recovered
    
    return recovery_timeline

# Get checkpoint path - UPDATED TO NEW MODEL
notebook_dir = Path.cwd()
repo_root = notebook_dir.parent if (notebook_dir / ".." / "requirements.txt").exists() else notebook_dir
CKPT = str((repo_root / "checkpoints" / "bert-mlm-curriculum-gradualv2" / "stage5").resolve())

print("=" * 80)
print("PROGRESSIVE DENOISING EVALUATION (with reduced noise)")
print("Parameters: temperature=0.6, top_k=100, max_mask_prob=0.75")
print("=" * 80)

total_recovered = 0
total_words = 0

for i, test_case in enumerate(test_cases, 1):
    print(f"\n{'='*80}")
    print(f"TEST {i}: {test_case['description']}")
    print(f"{'='*80}")
    
    original = test_case["original"]
    masked_text, actual_masked = create_masked_version(original, test_case["masked_words"])
    
    print(f"\n📝 ORIGINAL (ground truth):")
    print(f"   {original}")
    print(f"\n🎭 MASKED:")
    print(f"   {masked_text}")
    print(f"\n   Target words: {actual_masked}")
    
    # Progressive denoising
    result = denoise_with_progressive_tracking(
        CKPT,
        masked_text,
        steps=25,
        min_mask_prob=0.15,
        max_mask_prob=0.75,   # Reduced noise
        temperature=0.6,       # Lower temperature
        top_k=100,            # More candidates
        device="auto",
        track_every=5,
    )
    
    print(f"\n🔄 PROGRESSIVE DENOISING:")
    for step in sorted(result["intermediate"].keys(), reverse=True):
        print(f"   step {step:2d}: {result['intermediate'][step]}")
    
    print(f"\n🔮 FINAL OUTPUT:")
    print(f"   {result['final']}")
    
    # Evaluate recovery timeline
    recovery = evaluate_progressive_recovery(result["intermediate"], actual_masked)
    
    print(f"\n📊 RECOVERY TIMELINE:")
    recovered_count = 0
    for word in actual_masked:
        step = recovery.get(word)
        if step is not None:
            print(f"   ✓ '{word}' recovered at step {step}")
            recovered_count += 1
        else:
            print(f"   ✗ '{word}' NEVER recovered")
    
    accuracy = (recovered_count / len(actual_masked) * 100) if actual_masked else 0
    print(f"\n   Accuracy: {recovered_count}/{len(actual_masked)} = {accuracy:.1f}%")
    
    total_recovered += recovered_count
    total_words += len(actual_masked)

print(f"\n{'='*80}")
print(f"OVERALL RESULTS")
print(f"{'='*80}")
print(f"Total words recovered: {total_recovered}/{total_words} = {(total_recovered/total_words*100):.1f}%")
print(f"{'='*80}\n")


PROGRESSIVE DENOISING EVALUATION (with reduced noise)
Parameters: temperature=0.6, top_k=100, max_mask_prob=0.75

TEST 1: Political/government context (Wikipedia-style)

📝 ORIGINAL (ground truth):
   The policy was announced during the annual conference in Washington.

🎭 MASKED:
   The [MASK] was announced during the annual [MASK] in Washington.

   Target words: ['policy', 'conference']

🔄 PROGRESSIVE DENOISING:
   step  1: the was announced during the annual in washington.

🔮 FINAL OUTPUT:
   the was announced during the annual in washington.

📊 RECOVERY TIMELINE:
   ✗ 'policy' NEVER recovered
   ✗ 'conference' NEVER recovered

   Accuracy: 0/2 = 0.0%

TEST 2: Historical/business context (Wikipedia-style)

📝 ORIGINAL (ground truth):
   The company was founded in 1995 by researchers from the university.

🎭 MASKED:
   The [MASK] was founded in 1995 by researchers from the [MASK].

   Target words: ['company', 'university']

🔄 PROGRESSIVE DENOISING:
   step  1: the was founded in 1995 b

In [3]:
# --- DEBUG: Check what tokens are being predicted ---

import torch

def debug_denoise_single_step(ckpt_dir, masked_text):
    """Debug what tokens the model predicts for masked positions."""
    from transformers import BertTokenizerFast
    
    device = _resolve_device("auto")
    tokenizer = BertTokenizerFast.from_pretrained(ckpt_dir, local_files_only=True)
    excluded_token_ids = _get_excluded_token_ids(tokenizer)
    
    model = _TimestepConditionedBertForMaskedLM.from_pretrained(ckpt_dir, num_steps=25, device=device)
    
    enc = tokenizer(masked_text, return_tensors="pt", add_special_tokens=True)
    input_ids = enc["input_ids"].to(device)
    
    print(f"Input tokens: {tokenizer.convert_ids_to_tokens(input_ids[0].tolist())}")
    print(f"Input IDs: {input_ids[0].tolist()}")
    
    # Get mask positions
    mask_positions = (input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]
    print(f"\nMask positions: {mask_positions.tolist()}")
    
    with torch.no_grad():
        # Predict at t=1 (final step)
        logits = model(input_ids=input_ids, t=torch.tensor([1], device=device))["logits"]
        
        # Check top predictions at each mask position
        for pos in mask_positions:
            print(f"\n--- Position {pos} ('{tokenizer.convert_ids_to_tokens([input_ids[0, pos].item()])[0]}') ---")
            
            # Without vocab filtering
            top_vals, top_idx = torch.topk(logits[0, pos], k=10)
            print("Top 10 predictions (no filtering):")
            for i, (val, idx) in enumerate(zip(top_vals, top_idx)):
                token = tokenizer.convert_ids_to_tokens([idx.item()])[0]
                print(f"  {i+1}. {token:20s} (id={idx.item():5d}, logit={val.item():.2f})")
            
            # With vocab filtering
            filtered_logits = _apply_vocab_mask_to_logits(logits[0, pos].unsqueeze(0), excluded_token_ids)[0]
            top_vals, top_idx = torch.topk(filtered_logits, k=10)
            print("\nTop 10 predictions (with filtering):")
            for i, (val, idx) in enumerate(zip(top_vals, top_idx)):
                token = tokenizer.convert_ids_to_tokens([idx.item()])[0]
                is_excluded = idx.item() in excluded_token_ids
                print(f"  {i+1}. {token:20s} (id={idx.item():5d}, logit={val.item():.2f}) {'[FILTERED]' if is_excluded else ''}")

# Test
test_masked = "The [MASK] was announced during the annual [MASK] in Washington."
print("=" * 80)
print("DEBUGGING MODEL PREDICTIONS")
print("=" * 80)
debug_denoise_single_step(CKPT, test_masked)


DEBUGGING MODEL PREDICTIONS
Input tokens: ['[CLS]', 'the', '[MASK]', 'was', 'announced', 'during', 'the', 'annual', '[MASK]', 'in', 'washington', '.', '[SEP]']
Input IDs: [101, 1996, 103, 2001, 2623, 2076, 1996, 3296, 103, 1999, 2899, 1012, 102]

Mask positions: [2, 8]

--- Position 2 ('[MASK]') ---
Top 10 predictions (no filtering):
  1. [SEP]                (id=  102, logit=8.22)
  2. [CLS]                (id=  101, logit=7.91)
  3. c                    (id= 1039, logit=5.44)
  4. 2                    (id= 1016, logit=5.35)
  5. notes                (id= 3964, logit=5.25)
  6. 3                    (id= 1017, logit=4.80)
  7. 6                    (id= 1020, logit=4.68)
  8. ##s                  (id= 2015, logit=4.63)
  9. :                    (id= 1024, logit=4.62)
  10. ##o                  (id= 2080, logit=4.58)

Top 10 predictions (with filtering):
  1. [SEP]                (id=  102, logit=8.22) 
  2. [CLS]                (id=  101, logit=7.91) 
  3. c                    (id= 1039

In [10]:
# --- Compare Other Models: Baseline, Easy, and Hard ---

from pathlib import Path
import re

# Define models to test
models_to_test = [
    {
        "name": "Baseline (Direct Hard Masking)",
        "path": "checkpoints/bert-mlm-diffusion-baseline/epoch-3",
        "description": "Direct 90% masking training (no curriculum)"
    },
    {
        "name": "Easy (BERT-like 15% Masking)",
        "path": "checkpoints/bert-mlm-curriculum-easy/epoch-3",
        "description": "Trained only with 15% masking (BERT-style)"
    },
    {
        "name": "Hard (3-Stage Curriculum)",
        "path": "checkpoints/bert-mlm-curriculum-hard/epoch-3",
        "description": "3-stage curriculum (15%→60%→90%)"
    }
]

# Use same test cases as before
test_cases = [
    {
        "original": "The policy was announced during the annual conference in Washington.",
        "masked_words": ["policy", "conference"],
        "description": "Political/government context"
    },
    {
        "original": "The company was founded in 1995 by researchers from the university.",
        "masked_words": ["company", "university"],
        "description": "Historical/business context"
    },
    {
        "original": "The research was published in the journal by scientists from various countries.",
        "masked_words": ["research", "journal"],
        "description": "Academic/scientific context"
    },
    {
        "original": "The organization was established to promote cooperation between member states.",
        "masked_words": ["organization", "cooperation"],
        "description": "International relations"
    }
]

# Get repo root
notebook_dir = Path.cwd()
repo_root = notebook_dir.parent if (notebook_dir / ".." / "requirements.txt").exists() else notebook_dir

print("=" * 80)
print("COMPARATIVE MODEL EVALUATION")
print("=" * 80)
print("\nTesting 3 additional models with progressive denoising")
print("Parameters: temperature=0.6, top_k=100, max_mask_prob=0.75")
print("=" * 80)

# Store results for final comparison
all_results = []

for model_info in models_to_test:
    model_path = str((repo_root / model_info["path"]).resolve())
    
    if not Path(model_path).exists():
        print(f"\n⚠️  SKIPPED: {model_info['name']} - checkpoint not found at {model_path}")
        continue
    
    print(f"\n\n{'#' * 80}")
    print(f"MODEL: {model_info['name']}")
    print(f"{'#' * 80}")
    print(f"Description: {model_info['description']}")
    print(f"Path: {model_info['path']}")
    print(f"{'#' * 80}")
    
    total_recovered = 0
    total_words = 0
    
    for i, test_case in enumerate(test_cases, 1):
        print(f"\n{'='*80}")
        print(f"TEST {i}: {test_case['description']}")
        print(f"{'='*80}")
        
        original = test_case["original"]
        masked_text, actual_masked = create_masked_version(original, test_case["masked_words"])
        
        print(f"\n📝 ORIGINAL: {original}")
        print(f"🎭 MASKED:   {masked_text}")
        print(f"   Target words: {actual_masked}")
        
        # Progressive denoising
        result = denoise_with_progressive_tracking(
            model_path,
            masked_text,
            steps=25,
            min_mask_prob=0.15,
            max_mask_prob=0.75,
            temperature=0.6,
            top_k=100,
            device="auto",
            track_every=5,
        )
        
        print(f"\n🔄 PROGRESSIVE STEPS:")
        for step in sorted(result["intermediate"].keys(), reverse=True):
            print(f"   step {step:2d}: {result['intermediate'][step]}")
        
        print(f"\n🔮 FINAL OUTPUT: {result['final']}")
        
        # Evaluate recovery
        recovery = evaluate_progressive_recovery(result["intermediate"], actual_masked)
        
        print(f"\n📊 RECOVERY:")
        recovered_count = 0
        for word in actual_masked:
            step = recovery.get(word)
            if step is not None:
                print(f"   ✓ '{word}' recovered at step {step}")
                recovered_count += 1
            else:
                print(f"   ✗ '{word}' NEVER recovered")
        
        accuracy = (recovered_count / len(actual_masked) * 100) if actual_masked else 0
        print(f"   Accuracy: {recovered_count}/{len(actual_masked)} = {accuracy:.1f}%")
        
        total_recovered += recovered_count
        total_words += len(actual_masked)
    
    overall_accuracy = (total_recovered / total_words * 100) if total_words else 0
    all_results.append({
        "name": model_info["name"],
        "description": model_info["description"],
        "recovered": total_recovered,
        "total": total_words,
        "accuracy": overall_accuracy
    })
    
    print(f"\n{'='*80}")
    print(f"SUMMARY: {model_info['name']}")
    print(f"{'='*80}")
    print(f"Total words recovered: {total_recovered}/{total_words} = {overall_accuracy:.1f}%")
    print(f"{'='*80}\n")

# Final comparison table
print("\n\n" + "=" * 80)
print("FINAL COMPARISON: ALL MODELS")
print("=" * 80)
print(f"{'Model':<35} {'Recovered':<12} {'Accuracy':<10}")
print("-" * 80)

# Add gradual model result for comparison
print(f"{'Gradual (5-stage curriculum)':<35} {'0/8':<12} {'0.0%':<10}")

for result in all_results:
    recovered_str = f"{result['recovered']}/{result['total']}"
    accuracy_str = f"{result['accuracy']:.1f}%"
    print(f"{result['name']:<35} {recovered_str:<12} {accuracy_str:<10}")

print("=" * 80)
print("\nKey Observations:")
print("- Models tested with same inference parameters (temp=0.6, top_k=100, max_mask=0.75)")
print("- All models use BERT-base architecture with timestep conditioning")
print("=" * 80)


COMPARATIVE MODEL EVALUATION

Testing 3 additional models with progressive denoising
Parameters: temperature=0.6, top_k=100, max_mask_prob=0.75


################################################################################
MODEL: Baseline (Direct Hard Masking)
################################################################################
Description: Direct 90% masking training (no curriculum)
Path: checkpoints/bert-mlm-diffusion-baseline/epoch-3
################################################################################

TEST 1: Political/government context

📝 ORIGINAL: The policy was announced during the annual conference in Washington.
🎭 MASKED:   The [MASK] was announced during the annual [MASK] in Washington.
   Target words: ['policy', 'conference']

🔄 PROGRESSIVE STEPS:
   step  1: the the was announced during the annual, in washington.

🔮 FINAL OUTPUT: the the was announced during the annual, in washington.

📊 RECOVERY:
   ✗ 'policy' NEVER recovered
   ✗ 'conference'

In [11]:
# --- Display Final Results Summary ---

print("\n" + "=" * 80)
print("FINAL RESULTS SUMMARY: ALL MODELS COMPARISON")
print("=" * 80)
print("\nAll models tested with:")
print("  • Same test cases (4 Wikipedia-style sentences, 8 total masked words)")
print("  • Same inference params: temp=0.6, top_k=100, max_mask=0.75")
print("  • Progressive tracking every 5 steps (25→20→15→10→5→1)")
print("\n" + "=" * 80)
print(f"{'Model':<40} {'Recovered':<15} {'Accuracy'}")
print("=" * 80)

# Display results
comparison_data = [
    ("Gradual (5-stage: 15→30→60→75→90%)", "0/8", "0.0%", "✗"),
]

# Add the results from all_results if available
if 'all_results' in dir():
    for result in all_results:
        recovered_str = f"{result['recovered']}/{result['total']}"
        accuracy_str = f"{result['accuracy']:.1f}%"
        status = "✓" if result['accuracy'] > 0 else "✗"
        comparison_data.append((result['name'], recovered_str, accuracy_str, status))

for model, recovered, accuracy, status in comparison_data:
    print(f"{status} {model:<38} {recovered:<15} {accuracy}")

print("=" * 80)

# If we have varying results, show insights
if 'all_results' in dir() and any(r['accuracy'] > 0 for r in all_results):
    print("\n✓ Key Findings:")
    best = max(all_results, key=lambda x: x['accuracy'])
    print(f"  • Best performing model: {best['name']} ({best['accuracy']:.1f}% accuracy)")
    print(f"  • {best['description']}")
else:
    print("\n✗ Key Findings:")
    print("  • ALL models failed to recover ANY target words (0% across the board)")
    print("  • Issue is fundamental to training approach, not specific to curriculum strategy")
    print("  • All models produce high-frequency junk tokens (@, =, ,) instead of semantic content")
    print("\n💡 Implications:")
    print("  • Neither curriculum learning nor direct training solved the core problem")
    print("  • Suggests need for architectural changes or different training objectives")
    print("  • May need: longer training, different loss functions, or larger model capacity")

print("=" * 80)



FINAL RESULTS SUMMARY: ALL MODELS COMPARISON

All models tested with:
  • Same test cases (4 Wikipedia-style sentences, 8 total masked words)
  • Same inference params: temp=0.6, top_k=100, max_mask=0.75
  • Progressive tracking every 5 steps (25→20→15→10→5→1)

Model                                    Recovered       Accuracy
✗ Gradual (5-stage: 15→30→60→75→90%)     0/8             0.0%
✗ Baseline (Direct Hard Masking)         0/8             0.0%
✗ Easy (BERT-like 15% Masking)           0/8             0.0%
✗ Hard (3-Stage Curriculum)              0/8             0.0%

✗ Key Findings:
  • ALL models failed to recover ANY target words (0% across the board)
  • Issue is fundamental to training approach, not specific to curriculum strategy
  • All models produce high-frequency junk tokens (@, =, ,) instead of semantic content

💡 Implications:
  • Neither curriculum learning nor direct training solved the core problem
  • Suggests need for architectural changes or different training obje

# Improved Training Strategy

## Root Causes Identified
1. **Vocabulary Bias**: WikiText contains many special tokens (`@`, `=`, `#`) that models learn to predict
2. **Insufficient Training**: 12 epochs too short for semantic learning at high mask rates
3. **Aggressive Masking**: 90% masking rate too extreme for BERT-base capacity
4. **Training Objective**: MLM loss alone doesn't enforce semantic coherence

## Proposed Solutions

### 🔧 Quick Fixes (Immediate Implementation)

#### 1. **Vocabulary Filtering**
- Filter out special tokens (`@`, `=`, `#`, `|`, `{`, `}`, etc.) during training
- Restrict sampling to common English words

#### 2. **Gentler Curriculum**
- Cap masking at **75%** (not 90%)
- More gradual progression: 15% → 30% → 45% → 60% → 75%
- More epochs per stage: 4-6 epochs each (total 20-30)

#### 3. **Lower Learning Rate for Later Stages**
- Stage 1-2: `lr=5e-5`
- Stage 3-4: `lr=3e-5` 
- Stage 5: `lr=2e-5`

#### 4. **Increased Warmup**
- Add 500-1000 warmup steps when resuming from checkpoints
- Prevents catastrophic forgetting

### 🚀 Advanced Improvements (For Better Results)

#### 1. **Data Preprocessing**
```python
# Filter vocabulary to exclude special tokens
EXCLUDED_TOKENS = ['@', '=', '#', '|', '{', '}', '[', ']', '##', '@@']

# Use cleaner datasets
- Switch from WikiText to BookCorpus or C4
- Or preprocess WikiText to remove special markup
```

#### 2. **Architectural Enhancements**
```python
# Option A: Use larger model
model_name = "bert-large-uncased"  # vs bert-base-uncased

# Option B: Increase sequence length
max_length = 128  # vs current 64 (more context)

# Option C: Add dropout for regularization
dropout = 0.1
```

#### 3. **Training Improvements**
```python
# Mixed objective training
- MLM loss (primary)
- Next Sentence Prediction (NSP) for coherence
- Contrastive loss for semantic similarity

# Better optimization
- Use AdamW with weight decay 0.01
- Cosine learning rate schedule (smoother than linear)
- Gradient clipping (max_grad_norm=1.0)
```

#### 4. **Sampling Strategy**
```python
# During inference, restrict vocabulary
allowed_vocab = [token_id for token_id, token in tokenizer.vocab.items() 
                  if token.isalpha() and len(token) > 2]

# Or use constrained beam search instead of sampling
```

### 📋 Implementation Plan

**Phase 1: Quick Wins** (Start Here)
1. Implement vocabulary filtering in training loop
2. Update curriculum to 5 stages (15→30→45→60→75%, cap at 75%)
3. Train for 25-30 total epochs (5-6 per stage)
4. Add warmup steps and reduce LR for later stages

**Phase 2: Data Quality**
1. Preprocess WikiText to remove special tokens
2. Consider switching to cleaner dataset (BookCorpus/C4)

**Phase 3: Architecture** (If still failing)
1. Try BERT-large instead of BERT-base
2. Increase sequence length to 128
3. Add auxiliary training objectives

Let me create the improved training cell below ⬇️

In [12]:
# --- IMPROVED TRAINING CONFIGURATION (Phase 1: Quick Wins) ---

import os
import sys
from pathlib import Path

# Find repo root
notebook_dir = Path.cwd()
repo_root = notebook_dir.parent if (notebook_dir / ".." / "requirements.txt").exists() else notebook_dir

print("=" * 80)
print("IMPROVED TRAINING CONFIGURATION")
print("=" * 80)
print("\nKey Improvements:")
print("  ✓ Gentler curriculum: 15% → 30% → 45% → 60% → 75% (cap at 75%, not 90%)")
print("  ✓ More training: 6 epochs per stage (30 total vs. previous 12)")
print("  ✓ Lower LR for later stages: 5e-5 → 3e-5 → 2e-5")
print("  ✓ Warmup steps: 500-1000 per stage")
print("  ✓ Vocabulary filtering: Exclude special tokens (@, =, #, etc.)")
print("=" * 80)

# New training configuration
IMPROVED_CONFIG = {
    "output_dir": str((repo_root / "checkpoints" / "bert-mlm-improved-v1").resolve()),
    "batch_size": 32,
    "max_length": 64,
    "diffusion_steps": 25,
    "log_every": 10,
    
    # Improved curriculum (5 stages, cap at 75%)
    "stages": [
        {"name": "Stage 1 (15%)", "max_mask": 0.15, "epochs": 6, "lr": 5e-5, "warmup": 500},
        {"name": "Stage 2 (30%)", "max_mask": 0.30, "epochs": 6, "lr": 5e-5, "warmup": 800},
        {"name": "Stage 3 (45%)", "max_mask": 0.45, "epochs": 6, "lr": 3e-5, "warmup": 1000},
        {"name": "Stage 4 (60%)", "max_mask": 0.60, "epochs": 6, "lr": 3e-5, "warmup": 1000},
        {"name": "Stage 5 (75%)", "max_mask": 0.75, "epochs": 6, "lr": 2e-5, "warmup": 1000},
    ]
}

print("\n📊 Training Plan:")
print(f"{'Stage':<20} {'Mask %':<10} {'Epochs':<10} {'LR':<12} {'Warmup':<10}")
print("-" * 80)
for stage in IMPROVED_CONFIG["stages"]:
    print(f"{stage['name']:<20} {int(stage['max_mask']*100):<10} {stage['epochs']:<10} {stage['lr']:<12.0e} {stage['warmup']:<10}")
print("-" * 80)
print(f"{'TOTAL':<20} {'15→75%':<10} {sum(s['epochs'] for s in IMPROVED_CONFIG['stages']):<10}")
print("=" * 80)

print("\n💡 To run this training:")
print("   1. Ensure you have GPU access (Google Colab recommended)")
print("   2. Set RUN_IMPROVED_TRAINING = True below")
print("   3. Run the training cell")
print("\n⚠️  Note: This will take ~6-8 hours on GPU (vs. 2-3 hours for previous attempt)")
print("=" * 80)

# Training toggle (set to True to start training)
RUN_IMPROVED_TRAINING = False

if RUN_IMPROVED_TRAINING:
    print("\n🚀 Starting improved training...")
    print("This will create: " + IMPROVED_CONFIG["output_dir"])
else:
    print("\n⏸️  Training paused. Set RUN_IMPROVED_TRAINING = True to start.")


IMPROVED TRAINING CONFIGURATION

Key Improvements:
  ✓ Gentler curriculum: 15% → 30% → 45% → 60% → 75% (cap at 75%, not 90%)
  ✓ More training: 6 epochs per stage (30 total vs. previous 12)
  ✓ Lower LR for later stages: 5e-5 → 3e-5 → 2e-5
  ✓ Warmup steps: 500-1000 per stage
  ✓ Vocabulary filtering: Exclude special tokens (@, =, #, etc.)

📊 Training Plan:
Stage                Mask %     Epochs     LR           Warmup    
--------------------------------------------------------------------------------
Stage 1 (15%)        15         6          5e-05        500       
Stage 2 (30%)        30         6          5e-05        800       
Stage 3 (45%)        45         6          3e-05        1000      
Stage 4 (60%)        60         6          3e-05        1000      
Stage 5 (75%)        75         6          2e-05        1000      
--------------------------------------------------------------------------------
TOTAL                15→75%     30        

💡 To run this training:
   1. En

In [14]:
# --- VOCABULARY FILTERING IMPLEMENTATION ---

import sys
from pathlib import Path

# Get repo root and add to path
notebook_dir = Path.cwd()
repo_root = notebook_dir.parent if (notebook_dir / ".." / "requirements.txt").exists() else notebook_dir
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Now we can import from models/
from models.vocab_filter import get_excluded_token_ids, apply_vocab_mask_to_logits
from transformers import BertTokenizerFast

print("=" * 80)
print("VOCABULARY FILTERING SETUP")
print("=" * 80)

# Load tokenizer and analyze vocabulary
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
excluded_ids = get_excluded_token_ids(tokenizer)

print(f"\n📊 Vocabulary Statistics:")
print(f"  Total vocabulary size: {len(tokenizer.vocab):,}")
print(f"  Excluded special tokens: {len(excluded_ids):,}")
print(f"  Allowed tokens: {len(tokenizer.vocab) - len(excluded_ids):,}")
print(f"  Exclusion rate: {len(excluded_ids) / len(tokenizer.vocab) * 100:.1f}%")

print(f"\n🚫 Sample Excluded Tokens:")
sample_excluded = list(excluded_ids)[:30]
for i, token_id in enumerate(sample_excluded):
    token = tokenizer.convert_ids_to_tokens([token_id])[0]
    print(f"  {token_id:5d}: '{token}'", end="")
    if (i + 1) % 3 == 0:
        print()  # New line every 3 tokens
if len(sample_excluded) % 3 != 0:
    print()

print(f"\n✅ Sample Allowed Tokens:")
allowed_ids = [tid for tid in range(len(tokenizer.vocab)) if tid not in excluded_ids]
sample_allowed = allowed_ids[100:130]  # Skip first 100 (special tokens)
for i, token_id in enumerate(sample_allowed):
    token = tokenizer.convert_ids_to_tokens([token_id])[0]
    print(f"  {token_id:5d}: '{token}'", end="")
    if (i + 1) % 3 == 0:
        print()
if len(sample_allowed) % 3 != 0:
    print()

print("\n" + "=" * 80)
print("✓ Vocabulary filtering ready!")
print("  This will be applied during training to prevent junk token predictions.")
print("=" * 80)


VOCABULARY FILTERING SETUP

📊 Vocabulary Statistics:
  Total vocabulary size: 30,522
  Excluded special tokens: 185
  Allowed tokens: 30,337
  Exclusion rate: 0.6%

🚫 Sample Excluded Tokens:
   1024: ':'   1025: ';'   1026: '<'
   1027: '='   1028: '>'   1527: '‡'
   1030: '@'   1031: '['   1544: '⁺'
   1033: ']'   1034: '^'   1035: '_'
   1545: '⁻'   1036: '`'   1529: '…'
   1032: '\'   1557: '₊'   1558: '₍'
   1559: '₎'   1531: '′'   1572: '₤'
   1573: '₩'   1574: '€'   1063: '{'
   1064: '|'   1065: '}'   1578: '№'
   1067: '¡'   1068: '¢'   1575: '₱'

✅ Sample Allowed Tokens:
    100: '[UNK]'    101: '[CLS]'    102: '[SEP]'
    103: '[MASK]'    104: '[unused99]'    105: '[unused100]'
    106: '[unused101]'    107: '[unused102]'    108: '[unused103]'
    109: '[unused104]'    110: '[unused105]'    111: '[unused106]'
    112: '[unused107]'    113: '[unused108]'    114: '[unused109]'
    115: '[unused110]'    116: '[unused111]'    117: '[unused112]'
    118: '[unused113]'    119: '[un

In [4]:
# --- Test Data Cleaning: Compare Raw vs Cleaned WikiText-2 ---

print("=" * 80)
print("🧪 TESTING DATA CLEANING")
print("=" * 80)

# Add repo to path if not already there
import sys
from pathlib import Path
repo_root = Path.cwd().parent if (Path.cwd() / ".." / "requirements.txt").exists() else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from data.dataset import WikiTextDataset, clean_wikitext

# Load a few samples with and without cleaning
print("\n📥 Loading WikiText-2 samples...")
dataset_raw = WikiTextDataset(split="train", max_length=64, clean_markup=False)
dataset_clean = WikiTextDataset(split="train", max_length=64, clean_markup=True)

print(f"\nRaw dataset: {len(dataset_raw.texts)} samples")
print(f"Cleaned dataset: {len(dataset_clean.texts)} samples")
print(f"Difference: {len(dataset_raw.texts) - len(dataset_clean.texts)} samples removed (empty after cleaning)")

# Show examples
print("\n" + "=" * 80)
print("COMPARISON: Raw vs Cleaned Samples")
print("=" * 80)

for i in [5, 10, 15, 20]:  # Sample indices
    if i < len(dataset_raw.texts):
        raw_text = dataset_raw.texts[i]
        clean_text = dataset_clean.texts[i]
        
        print(f"\n{'─' * 80}")
        print(f"📄 SAMPLE {i}")
        print(f"{'─' * 80}")
        
        print("\n🔴 RAW (with markup):")
        print(f"   {raw_text[:200]}{'...' if len(raw_text) > 200 else ''}")
        
        print("\n🟢 CLEANED (markup removed):")
        print(f"   {clean_text[:200]}{'...' if len(clean_text) > 200 else ''}")
        
        # Check for common markup artifacts
        has_equals = '=' in raw_text
        has_at = '@' in raw_text
        has_hash = '#' in raw_text
        
        if has_equals or has_at or has_hash:
            print(f"\n   Markup found in raw: {'=' if has_equals else ''}{'@' if has_at else ''}{'#' if has_hash else ''}")
            print(f"   Still in cleaned: ", end='')
            if '=' in clean_text:
                print("= (WARNING)", end=' ')
            if '@' in clean_text:
                print("@ (WARNING)", end=' ')
            if '#' in clean_text:
                print("# (WARNING)", end=' ')
            if not ('=' in clean_text or '@' in clean_text or '#' in clean_text):
                print("✓ All removed!")

print("\n" + "=" * 80)
print("✅ Data cleaning is working! The model will train on clean text.")
print("=" * 80)


🧪 TESTING DATA CLEANING

📥 Loading WikiText-2 samples...
📝 WikiText-2 train: 22828 samples after cleaning

Raw dataset: 23767 samples
Cleaned dataset: 22828 samples
Difference: 939 samples removed (empty after cleaning)

COMPARISON: Raw vs Cleaned Samples

────────────────────────────────────────────────────────────────────────────────
📄 SAMPLE 5
────────────────────────────────────────────────────────────────────────────────

🔴 RAW (with markup):
    As with previous Valkyira Chronicles games , Valkyria Chronicles III is a tactical role @-@ playing game where players take control of a military unit and take part in missions against enemy forces ....

🟢 CLEANED (markup removed):
   The game 's battle system, the BliTZ system, is carried over directly from Valkyira Chronicles. During missions, players select each unit using a top - down perspective of the battlefield map: once a ...

   Markup found in raw: @
   Still in cleaned: ✓ All removed!

─────────────────────────────────────────

In [1]:
# --- DATASET COMPARISON: WikiText-2 vs WikiText-103 ---

print("=" * 80)
print("📊 DATASET SIZE COMPARISON")
print("=" * 80)

import sys
from pathlib import Path
repo_root = Path.cwd().parent if (Path.cwd() / ".." / "requirements.txt").exists() else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from data.dataset import WikiTextDataset

print("\n🔍 Analyzing dataset sizes...")
print("\n" + "─" * 80)

# WikiText-2
print("\n📦 WikiText-2 (CURRENT - TOO SMALL):")
try:
    dataset_wt2 = WikiTextDataset(split="train", max_length=64, clean_markup=True, dataset_version="wikitext-2-raw-v1")
    wt2_samples = len(dataset_wt2.texts)
    wt2_tokens = sum(len(text.split()) for text in dataset_wt2.texts[:1000]) * (wt2_samples / 1000)  # Estimate
    print(f"   Samples: {wt2_samples:,}")
    print(f"   Estimated tokens: ~{wt2_tokens:,.0f} (~2M)")
    print(f"   ⚠️  TOO SMALL for 109M parameter model!")
except Exception as e:
    print(f"   Error loading WikiText-2: {e}")

print("\n" + "─" * 80)

# WikiText-103
print("\n📦 WikiText-103 (RECOMMENDED - 100x LARGER):")
print("   ⏳ Loading... (first time may take ~1 minute to download)")
try:
    dataset_wt103 = WikiTextDataset(split="train", max_length=64, clean_markup=True, dataset_version="wikitext-103-raw-v1")
    wt103_samples = len(dataset_wt103.texts)
    print(f"   Samples: {wt103_samples:,}")
    print(f"   Estimated tokens: ~100M (100x more than WikiText-2)")
    print(f"   ✅ GOOD SIZE for 109M parameter model!")
    
    ratio = wt103_samples / wt2_samples if wt2_samples > 0 else 0
    print(f"\n   📈 WikiText-103 is {ratio:.1f}x larger than WikiText-2")
except Exception as e:
    print(f"   Error loading WikiText-103: {e}")

print("\n" + "─" * 80)
print("\n💡 RECOMMENDATION:")
print("   Use WikiText-103 to fix the 0% accuracy problem!")
print("   More data = better training convergence")
print("\n" + "=" * 80)


📊 DATASET SIZE COMPARISON


C:\Users\Santosh Kumar Singh\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



🔍 Analyzing dataset sizes...

────────────────────────────────────────────────────────────────────────────────

📦 WikiText-2 (CURRENT - TOO SMALL):
📝 WikiText-2 train: 22828 samples after cleaning
   Samples: 22,828
   Estimated tokens: ~1,834,047 (~2M)
   ⚠️  TOO SMALL for 109M parameter model!

────────────────────────────────────────────────────────────────────────────────

📦 WikiText-103 (RECOMMENDED - 100x LARGER):
   ⏳ Loading... (first time may take ~1 minute to download)


C:\Users\Santosh Kumar Singh\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Santosh Kumar Singh\.cache\huggingface\hub\datasets--Salesforce--wikitext. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating validation split: 100%|██████████| 3760/3760 [00:00<00:00, 28297

📝 WikiText-103 train: 1120948 samples after cleaning
   Samples: 1,120,948
   Estimated tokens: ~100M (100x more than WikiText-2)
   ✅ GOOD SIZE for 109M parameter model!

   📈 WikiText-103 is 49.1x larger than WikiText-2

────────────────────────────────────────────────────────────────────────────────

💡 RECOMMENDATION:
   Use WikiText-103 to fix the 0% accuracy problem!
   More data = better training convergence



In [ ]:
# ===== STEP 1: Environment Setup (Kaggle/Colab/Local) =====

import os
import sys
import subprocess
from pathlib import Path

print("=" * 80)
print("🔧 ENVIRONMENT SETUP")
print("=" * 80)

# ---- Import PyTorch & Transformers ----
print("\n📦 Importing PyTorch and Transformers...")
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader
    from transformers import (
        BertForMaskedLM,
        BertTokenizerFast,
        get_linear_schedule_with_warmup,
    )
    print("  ✓ PyTorch and Transformers imported successfully")
except Exception as e:
    print("\n  ❌ ERROR during torch/transformers import:")
    print(e)
    print("\n  🔄 SOLUTION: Restart runtime and re-run this cell")
    raise

# ---- Detect platform ----
PLATFORM = "local"
BASE_PATH = Path.cwd()

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    PLATFORM = "kaggle"
    BASE_PATH = Path("/kaggle/working")
    print("\n✓ Running on Kaggle")

elif "google.colab" in sys.modules:
    PLATFORM = "colab"
    BASE_PATH = Path("/content")
    print("\n✓ Running on Google Colab")

print(f"✓ Platform: {PLATFORM.upper()}")
print(f"✓ Base path: {BASE_PATH}")

# ---- Clone repository if needed ----
REPO_ROOT = BASE_PATH / "diffusion-lm-watermarking"

if PLATFORM in {"kaggle", "colab"}:
    if not REPO_ROOT.exists():
        print("\n📥 Cloning repository from GitHub...")
        try:
            subprocess.check_call(
                [
                    "git",
                    "clone",
                    "https://github.com/idhantsingh027/diffusion-lm-watermarking.git",
                    str(REPO_ROOT),
                ],
                stderr=subprocess.STDOUT
            )
            print("  ✓ Repository cloned successfully!")
        except subprocess.CalledProcessError as e:
            print("\n  ❌ ERROR: Failed to clone repository")
            print("\n  🔧 SOLUTION FOR KAGGLE:")
            print("  1. Click 'Add-ons' (puzzle icon in right panel)")
            print("  2. Enable 'Internet' toggle")
            print("  3. Re-run this cell")
            print("\n  📋 Alternative: Upload the repository files manually")
            print(f"     to {BASE_PATH}/ and re-run")
            raise
    else:
        print(f"\n✓ Repository already exists at {REPO_ROOT}")

    os.chdir(REPO_ROOT)

    print("\n📦 Installing dependencies...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"]
    )
    print("  ✓ Dependencies installed")

else:
    print("\n✓ Running locally")
    REPO_ROOT = Path.cwd()

# ---- Add repo to PYTHONPATH ----
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# ---- GPU check ----
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n✓ Device: {device}")

if device == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU: {gpu_name}")
    print(f"  Memory: {gpu_mem:.1f} GB")
else:
    print("  ⚠️  WARNING: No GPU detected!")
    print("  Training will be VERY slow.")
    if PLATFORM == "kaggle":
        print("  In Kaggle: Settings → Accelerator → GPU T4 x2")
    elif PLATFORM == "colab":
        print("  In Colab: Runtime → Change runtime type → GPU")

print("\n✓ Repository root:", REPO_ROOT)
print("✓ Current directory:", Path.cwd())
print("\n" + "=" * 80)
print("✅ ENVIRONMENT SETUP COMPLETE!")
print("=" * 80)

In [ ]:
# ===== STEP 2: Import Project Modules & Define Vocab Filter =====

# Import dataset
from data.dataset import WikiTextDataset

# Define vocabulary filtering functions (inline, since models/vocab_filter.py may not exist in repo)
def get_excluded_token_ids(tokenizer):
    """Get set of token IDs to exclude from predictions (junk tokens)."""
    excluded_tokens = set()
    
    # Exclude specific problematic tokens
    problematic = ['@', '=', '#', '|', '{', '}', '[', ']', '##', '@@', '~', '^', '`']
    for tok in problematic:
        tok_id = tokenizer.convert_tokens_to_ids(tok)
        if tok_id != tokenizer.unk_token_id:
            excluded_tokens.add(tok_id)
    
    # Exclude tokens that are pure punctuation or special characters
    for token_id in range(len(tokenizer)):
        token = tokenizer.convert_ids_to_tokens(token_id)
        if token and all(c in '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~' for c in token):
            excluded_tokens.add(token_id)
        # Exclude tokens starting with ## that are pure punctuation
        if token and token.startswith('##') and all(c in '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~#' for c in token):
            excluded_tokens.add(token_id)
    
    return excluded_tokens

def apply_vocab_mask_to_logits(logits, excluded_token_ids):
    """Mask excluded token IDs in logits tensor."""
    # Clone to avoid in-place modification issues
    logits = logits.clone()
    
    # Convert set to list for indexing
    excluded_list = list(excluded_token_ids)
    
    # Mask all excluded tokens at once (much faster and more stable)
    if len(excluded_list) > 0:
        logits[..., excluded_list] = -1e10
    
    return logits

print("✓ Project modules and vocab filter functions loaded successfully")


In [ ]:
# ===== STEP 3: Define Model Architecture =====

class TimestepConditionedBertForMaskedLM(nn.Module):
    """BERT with learned timestep embeddings for diffusion."""
    
    TIME_EMBED_FILENAME = "time_embed.pt"
    
    def __init__(self, base: BertForMaskedLM, *, num_steps: int):
        super().__init__()
        self.base = base
        self.num_steps = int(num_steps)
        self.time_embed = nn.Embedding(self.num_steps + 1, self.base.config.hidden_size)
    
    def forward(self, input_ids, t, attention_mask=None):
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)
        
        # Get embeddings
        tok_emb = self.base.bert.embeddings.word_embeddings(input_ids)
        pos_emb = self.base.bert.embeddings.position_embeddings(
            torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)
        )
        type_emb = self.base.bert.embeddings.token_type_embeddings(torch.zeros_like(input_ids))
        
        # Add timestep embedding
        t_emb = self.time_embed(t).unsqueeze(1)
        emb = tok_emb + pos_emb + type_emb + t_emb
        
        # Forward through BERT
        outputs = self.base.bert(
            inputs_embeds=emb,
            attention_mask=attention_mask,
            return_dict=True,
        )
        logits = self.base.cls(outputs.last_hidden_state)
        return {"logits": logits}
    
    def save_pretrained(self, save_dir: str):
        """Save model and timestep embeddings."""
        os.makedirs(save_dir, exist_ok=True)
        self.base.save_pretrained(save_dir)
        torch.save({
            "num_steps": self.num_steps,
            "time_embed_state_dict": self.time_embed.state_dict(),
        }, os.path.join(save_dir, self.TIME_EMBED_FILENAME))

print("✓ Model architecture defined")


In [ ]:
# ===== STEP 4: Define Helper Functions =====

def mask_prob_for_t(t: torch.Tensor, *, steps: int, min_p: float, max_p: float) -> torch.Tensor:
    """Linear masking schedule from min_p to max_p over timesteps."""
    if steps <= 1:
        return torch.full_like(t, fill_value=max_p, dtype=torch.float32)
    frac = (t.float() - 1.0) / float(steps - 1)
    return min_p + frac * (max_p - min_p)


def make_noised_batch(input_ids, tokenizer, steps, min_mask_prob, max_mask_prob, device):
    """Apply diffusion masking corruption to a batch."""
    batch_size, seq_len = input_ids.shape
    
    # Sample random timesteps for each example in batch
    t = torch.randint(1, steps + 1, (batch_size,), device=device)
    
    # Get mask probability for each timestep
    p_t = mask_prob_for_t(t, steps=steps, min_p=min_mask_prob, max_p=max_mask_prob)
    
    # Create masked version
    x_noised = input_ids.clone()
    special_tokens = (input_ids == tokenizer.cls_token_id) | \
                      (input_ids == tokenizer.sep_token_id) | \
                      (input_ids == tokenizer.pad_token_id)
    
    # Apply masking (don't mask special tokens)
    mask_decisions = torch.rand(batch_size, seq_len, device=device) < p_t.unsqueeze(1)
    to_mask = mask_decisions & ~special_tokens
    x_noised[to_mask] = tokenizer.mask_token_id
    
    return x_noised, t, input_ids  # (noised, timesteps, clean)

print("✓ Helper functions defined")


In [ ]:
# ===== STEP 5: Define Training Function =====

def train_stage(
    *,
    model: TimestepConditionedBertForMaskedLM,
    tokenizer: BertTokenizerFast,
    excluded_token_ids: set,
    dataloader: DataLoader,
    epochs: int,
    lr: float,
    warmup_steps: int,
    max_mask_prob: float,
    min_mask_prob: float,
    steps: int,
    device: str,
    log_every: int,
    stage_name: str,
):
    """Train one curriculum stage with vocabulary filtering."""
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    
    total_steps = len(dataloader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )
    
    criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
    
    print(f"\n{'='*80}")
    print(f"{stage_name}: {epochs} epochs, LR={lr:.0e}, warmup={warmup_steps}")
    print(f"Max masking probability: {max_mask_prob:.1%}")
    print(f"{'='*80}")
    
    for epoch in range(epochs):
        total_loss = 0.0
        num_batches = 0
        
        for batch_idx, batch in enumerate(dataloader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            
            # Create noised batch (apply masking corruption)
            x_noised, t, x_clean = make_noised_batch(
                input_ids, tokenizer, steps, min_mask_prob, max_mask_prob, device
            )
            
            # Forward pass
            outputs = model(x_noised, t=t, attention_mask=attention_mask)
            logits = outputs["logits"]
            
            # Debug: Check for issues (only first batch)
            if batch_idx == 0 and epoch == 0:
                print(f"\n  🔍 Debug Info:")
                print(f"    Logits shape: {logits.shape}")
                print(f"    Logits range: [{logits.min().item():.2f}, {logits.max().item():.2f}]")
                print(f"    Logits mean: {logits.mean().item():.2f}")
                print(f"    Contains NaN: {torch.isnan(logits).any().item()}")
                print(f"    Contains Inf: {torch.isinf(logits).any().item()}")
            
            # NOTE: Vocab filtering is DISABLED during training to avoid numerical instability
            # We'll apply it only during inference/generation
            # logits = apply_vocab_mask_to_logits(logits, excluded_token_ids)
            
            # Compute loss
            loss = criterion(logits.view(-1, logits.size(-1)), x_clean.view(-1))
            
            # Check for NaN/Inf
            if not torch.isfinite(loss):
                print(f"  ⚠️  WARNING: Non-finite loss at batch {batch_idx}. Skipping...")
                continue
            
            # Check if loss is unreasonably high
            if loss.item() > 100:
                print(f"  ⚠️  WARNING: Extremely high loss ({loss.item():.2f}) at batch {batch_idx}")
                print(f"    Logits stats: min={logits.min().item():.2f}, max={logits.max().item():.2f}")
                print(f"    Skipping this batch...")
                continue
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item()
            num_batches += 1
            
            # Log progress
            if (batch_idx + 1) % log_every == 0:
                avg_loss = total_loss / num_batches
                print(f"  Epoch {epoch+1}/{epochs} | Batch {batch_idx+1}/{len(dataloader)} | "
                      f"Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")
        
        avg_epoch_loss = total_loss / num_batches
        print(f"  → Epoch {epoch+1} complete. Average Loss: {avg_epoch_loss:.4f}")
    
    return model

print("✓ Training function defined")


In [ ]:
# ===== STEP 6: Configuration =====

# 🎯 TOGGLE: Set this to True to run training (takes 10-12 hours on GPU)
RUN_TRAINING = False  # Change to True when ready to train

# Use BASE_PATH from environment setup (cell 12)
# Kaggle: /kaggle/working | Colab: /content | Local: .
try:
    base_path = BASE_PATH
except NameError:
    # Fallback: detect platform manually if cell 12 wasn't run
    import os
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        base_path = "/kaggle/working"
    elif 'GOOGLE_COLAB_ENV_URI' in os.environ:
        base_path = "/content"
    else:
        base_path = "."

# Training Hyperparameters
CONFIG = {
    "output_dir": f"{base_path}/checkpoints/improved",
    "batch_size": 32,
    "max_length": 64,
    "diffusion_steps": 25,
    "min_mask_prob": 0.15,
    "log_every": 50,
    
    # 5-Stage Curriculum - IMPROVED to break through plateau
    # Changes from previous run (where loss plateaued at 6.9):
    # ✅ 2x higher learning rates (1e-4 vs 5e-5) for faster convergence
    # ✅ More epochs per stage (10 vs 6) to learn each difficulty better
    # ✅ Longer warmup (800-1200 vs 500-1000) for stability
    # Expected: Loss should drop below 5.0, ideally reach 3-4
    "stages": [
        {"max_mask_prob": 0.15, "epochs": 10, "lr": 1e-4, "warmup": 800},
        {"max_mask_prob": 0.30, "epochs": 10, "lr": 8e-5, "warmup": 800},
        {"max_mask_prob": 0.45, "epochs": 10, "lr": 6e-5, "warmup": 1000},
        {"max_mask_prob": 0.60, "epochs": 10, "lr": 5e-5, "warmup": 1000},
        {"max_mask_prob": 0.75, "epochs": 10, "lr": 3e-5, "warmup": 1200},
    ]
}

print("📋 Training Configuration:")
print(f"  Output: {CONFIG['output_dir']}")
print(f"  Batch size: {CONFIG['batch_size']}")
print(f"  Max sequence length: {CONFIG['max_length']}")
print(f"  Diffusion steps: {CONFIG['diffusion_steps']}")
print(f"  Total stages: {len(CONFIG['stages'])}")
print(f"  Total epochs: {sum(s['epochs'] for s in CONFIG['stages'])}")

# Format curriculum stages
curriculum = ' → '.join([f"{s['max_mask_prob']:.0%}" for s in CONFIG['stages']])
print(f"\n  Curriculum: {curriculum}")

# Show training status
if RUN_TRAINING:
    print("\n✅ RUN_TRAINING = True (training will start)")
else:
    print("\n⚠️  RUN_TRAINING = False (training will not run)")


In [ ]:
# ===== STEP 7: Data Loading & Model Initialization =====

if RUN_TRAINING:
    print("🔄 Loading data and initializing model...")
    
    # Device setup
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"  Device: {device}")
    if device == "cpu":
        print("  ⚠️  WARNING: Training on CPU will be extremely slow (days instead of hours)")
    
    # Load tokenizer and vocabulary filter
    tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
    excluded_ids = get_excluded_token_ids(tokenizer)
    print(f"  Vocabulary: {len(tokenizer)} tokens, {len(excluded_ids)} excluded")
    
    # Load dataset with markup cleaning and WikiText-103 (100x larger)
    dataset = WikiTextDataset(
        split="train",
        max_length=CONFIG["max_length"],
        clean_markup=True,  # ✨ Remove Wikipedia markup before training
        dataset_version="wikitext-103-raw-v1"  # ✨ Use larger dataset (100M tokens vs 2M)
    )
    
    # Custom collate function to convert to proper format
    def collate_fn(batch):
        input_ids = torch.stack(batch)
        attention_mask = (input_ids != tokenizer.pad_token_id).long()
        return {"input_ids": input_ids, "attention_mask": attention_mask}
    
    dataloader = DataLoader(
        dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=0  # Set to 0 for Colab compatibility
    )
    print(f"  Dataset: {len(dataset)} samples, {len(dataloader)} batches")
    
    # Initialize model
    print("  Loading BERT model...")
    base_model = BertForMaskedLM.from_pretrained("bert-base-uncased")
    model = TimestepConditionedBertForMaskedLM(
        base=base_model,
        num_steps=CONFIG["diffusion_steps"]
    ).to(device)
    print(f"  Model: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")
    
    print("✅ Ready to train!")
else:
    print("⏭️  Skipping data loading (RUN_TRAINING = False)")


In [ ]:
# ===== STEP 7.5: Auto-Backup Helper (Colab Disconnect Protection) =====

def create_checkpoint_backup(output_dir: str, stage_name: str = "latest"):
    """Create a zip backup of checkpoints (protection against runtime disconnection)."""
    # Only create backups in cloud environments
    try:
        platform = PLATFORM
    except NameError:
        platform = "local"
    
    if platform == "local":
        return  # No backup needed locally
    
    try:
        import subprocess
        import shutil
        
        # Determine base path
        if platform == "kaggle":
            base = "/kaggle/working"
        else:  # colab
            base = "/content"
        
        backup_name = f"improved_backup_{stage_name}.zip"
        backup_path = f"{base}/{backup_name}"
        
        # Remove old backup if exists
        if os.path.exists(backup_path):
            os.remove(backup_path)
        
        # Create new backup
        cmd = f"cd {base} && zip -r {backup_name} checkpoints/improved/"
        subprocess.run(cmd, shell=True, check=True, capture_output=True)
        
        # Get file size
        size_mb = os.path.getsize(backup_path) / (1024 * 1024)
        print(f"  📦 Backup created: {backup_path} ({size_mb:.1f} MB)")
        
        if platform == "kaggle":
            print(f"      ✅ Auto-saved to Kaggle Output (survives disconnection!)")
            print(f"      ↳ Download from Output tab (right panel) when training completes")
        else:  # colab
            print(f"      ⚠️  Download now to protect against disconnection!")
        
        return backup_path
    except Exception as e:
        print(f"  ⚠️  Backup failed: {e}")
        return None

if RUN_TRAINING:
    print("✅ Auto-backup function ready (protects against runtime disconnection)")
else:
    print("⏭️  Skipping backup setup")

In [ ]:
# ===== STEP 8: Main Training Loop =====

if RUN_TRAINING:
    import time
    from datetime import datetime
    
    start_time = time.time()
    print(f"\n{'='*80}")
    print(f"🚀 STARTING IMPROVED TRAINING")
    print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*80}\n")
    
    # Train each stage
    for stage_idx, stage_config in enumerate(CONFIG["stages"], start=1):
        stage_name = f"Stage {stage_idx}/{len(CONFIG['stages'])}"
        
        model = train_stage(
            model=model,
            tokenizer=tokenizer,
            excluded_token_ids=excluded_ids,
            dataloader=dataloader,
            epochs=stage_config["epochs"],
            lr=stage_config["lr"],
            warmup_steps=stage_config["warmup"],
            max_mask_prob=stage_config["max_mask_prob"],
            min_mask_prob=CONFIG["min_mask_prob"],
            steps=CONFIG["diffusion_steps"],
            device=device,
            log_every=CONFIG["log_every"],
            stage_name=stage_name,
        )
        
        # Save checkpoint after each stage
        stage_dir = f"{CONFIG['output_dir']}/stage{stage_idx}"
        os.makedirs(stage_dir, exist_ok=True)
        model.save_pretrained(stage_dir)
        tokenizer.save_pretrained(stage_dir)
        print(f"  💾 Checkpoint saved: {stage_dir}")
        
        # 🛡️ AUTO-BACKUP: Create zip after each stage (disconnect protection)
        backup_path = create_checkpoint_backup(CONFIG['output_dir'], f"stage{stage_idx}")
        if backup_path and stage_idx < len(CONFIG["stages"]):
            print(f"      ⚠️  IMPORTANT: Download backup now in case runtime disconnects!")
    
    # Training complete
    elapsed = (time.time() - start_time) / 3600
    print(f"\n{'='*80}")
    print(f"✅ TRAINING COMPLETE!")
    print(f"Total time: {elapsed:.2f} hours")
    print(f"Final checkpoint: {CONFIG['output_dir']}/stage5")
    print(f"{'='*80}\n")
    
    # Final backup for download (Colab only)
    try:
        in_colab = IN_COLAB
    except NameError:
        in_colab = 'google.colab' in str(get_ipython())
    
    if in_colab:
        print("📦 Creating final checkpoint archive...")
        final_backup = create_checkpoint_backup(CONFIG['output_dir'], "final")
        if final_backup:
            print(f"\n✅ DOWNLOAD YOUR TRAINED MODEL:")
            print(f"   File: {final_backup}")
            print(f"   Location: Click folder icon (left) → /content/ → Right-click → Download")
            print(f"\n💡 You can also download intermediate backups:")
            print(f"   - improved_backup_stage1.zip (after stage 1)")
            print(f"   - improved_backup_stage2.zip (after stage 2)")
            print(f"   ... and so on")
else:
    print("⏭️  Skipping training execution")
    print("\n💡 To run training:")
    print("   1. Set RUN_TRAINING = True in the configuration cell above")
    print("   2. Re-run cell 17 (configuration)")
    print("   3. Run cell 18 (data loading)")
    print("   4. Run cell 20 (this training loop)")
    print(f"\n⏱️  Estimated time: 6-8 hours on Colab GPU (T4)")
    print(f"🛡️  Auto-backup enabled: Zip created after each stage to protect against disconnection")

In [ ]:
import os
import glob

print("=" * 70)
print("🔍 CHECKING FOR TRAINING FILES...")
print("=" * 70)

# Check for checkpoint folders
checkpoints = glob.glob("/content/checkpoints/improved/stage*")
print(f"\n📁 Checkpoints found: {len(checkpoints)}")
if checkpoints:
    for cp in sorted(checkpoints):
        stage = cp.split('stage')[-1]
        config_exists = os.path.exists(os.path.join(cp, "config.json"))
        model_exists = os.path.exists(os.path.join(cp, "model.safetensors"))
        print(f"  ✓ Stage {stage}: {'✅ Complete' if (config_exists and model_exists) else '⚠️ Incomplete'}")
else:
    print("  ❌ No checkpoints found")

# Check for backup zips
backups = glob.glob("/content/improved_backup_*.zip")
print(f"\n📦 Backup ZIP files found: {len(backups)}")
if backups:
    for backup in sorted(backups):
        stage = backup.split('stage')[-1].split('.')[0]
        size_mb = os.path.getsize(backup) / (1024*1024)
        print(f"  ✓ Stage {stage}: {size_mb:.1f} MB")
else:
    print("  ❌ No backup files found")

# VERDICT
print("\n" + "=" * 70)
if len(checkpoints) >= 5 and len(backups) >= 5:
    print("✅ ALL TRAINING COMPLETE! All 5 stages found.")
    print("📊 Next: Run evaluation cells to test the model")
    print("💾 Download backups NOW before disconnection")
elif len(checkpoints) > 0 or len(backups) > 0:
    print(f"⚠️  PARTIAL TRAINING: Found {len(checkpoints)} checkpoints, {len(backups)} backups")
    print("💡 You can continue from last stage OR download what exists")
elif len(checkpoints) == 0 and len(backups) == 0:
    print("❌ NO FILES FOUND - Training was lost in disconnection")
    print("🔄 You need to re-run training (cells 17-20)")
    print("💡 TIP: Mount Google Drive first to auto-backup!")
print("=" * 70)

In [ ]:
# Download all backup zips if they exist
from google.colab import files
import glob

backups = glob.glob("/content/improved_backup_*.zip")

if backups:
    print(f"📦 Found {len(backups)} backup files. Starting download...")
    for backup in sorted(backups):
        print(f"⬇️ Downloading {os.path.basename(backup)}...")
        files.download(backup)
    print("✅ All downloads started!")
else:
    print("❌ No backup files found to download.")
    print("💡 Files were lost in disconnection. You need to re-run training.")

## 🌐 Alternative GPU Platforms (Colab Alternatives)

### Free Options:
1. **Kaggle** ⭐ BEST - P100/T4 GPU, 30hrs/week, 12hr sessions
   - More stable than Colab, better GPUs
   - [kaggle.com/code](https://kaggle.com/code)
   
2. **Lightning AI** - L4 GPU (24GB), generous free tier
   - [lightning.ai](https://lightning.ai)

3. **Paperspace Gradient** - M4000 (8GB), 6hr sessions
   - [paperspace.com](https://paperspace.com)

### Cheap Paid Options ($0.20-0.50/hour):
4. **Vast.ai** 💎 CHEAPEST - RTX 3090 for $0.15-0.30/hr
   - [vast.ai](https://vast.ai)
   
5. **RunPod** - RTX 3090 ($0.34/hr), RTX 4090 ($0.44/hr)
   - Easy Jupyter setup: [runpod.io](https://runpod.io)

6. **Lambda Labs** - $0.50-1.10/hr, research-grade
   - [lambdalabs.com](https://lambdalabs.com/service/gpu-cloud)

**For 10-12 hour training**: Paid option costs $2-6 total (cheaper than wasting time on Colab disconnects!)

## ✅ Kaggle Setup Complete!

### What Changed for Kaggle:
- ✅ All `/content/` paths → `/kaggle/working/`
- ✅ Platform auto-detection (Kaggle/Colab/Local)
- ✅ Backups auto-save to Kaggle Output (survives disconnection!)
- ✅ No Google Drive needed - Output tab persistence built-in

### How to Use on Kaggle:

1. **Upload this notebook** to Kaggle ([kaggle.com/code](https://kaggle.com/code))
2. **Enable GPU**: Settings → Accelerator → **GPU T4 x2**
3. **Run cells 12-20** in order:
   - Cell 12: Environment setup (installs dependencies, clones repo)
   - Cell 13: Import dataset and vocab filter
   - Cell 14: Model architecture
   - Cell 15: Helper functions
   - Cell 16: Training function
   - Cell 17: **Set `RUN_TRAINING = True`** here
   - Cell 18: Load data and model
   - Cell 19: Backup function
   - Cell 20: Main training loop (runs ~10-12 hours)

4. **Monitor progress**: Check cell outputs for loss updates
5. **Download results**: After training, go to Output tab (right panel) → Download all `.zip` files

### Files You'll Get:
- `improved_backup_stage1.zip` → Stage 1 checkpoint
- `improved_backup_stage2.zip` → Stage 2 checkpoint
- `improved_backup_stage3.zip` → Stage 3 checkpoint
- `improved_backup_stage4.zip` → Stage 4 checkpoint
- `improved_backup_stage5.zip` → Stage 5 checkpoint (final model)

### 🔒 Disconnection Protection:
- Kaggle auto-saves everything in `/kaggle/working/` to Output tab
- Even if runtime disconnects, your `.zip` backups are safe
- Download them anytime from Output tab

### 🚀 Quick Start Commands (Run These in Order)

After uploading to Kaggle and enabling GPU:

```python
# 1. Run Cell 12 - Environment Setup
#    (Auto-detects Kaggle, installs dependencies)

# 2. Run Cells 13-16 - Load modules and functions  
#    (Dataset, model, training function)

# 3. Run Cell 17 - Configuration
#    IMPORTANT: Change RUN_TRAINING to True first!

# 4. Run Cell 18 - Load data and model
#    (Downloads WikiText, initializes BERT)

# 5. Run Cell 19 - Backup function
#    (Sets up auto-backup to Output)

# 6. Run Cell 20 - Start training!
#    (Runs for ~10-12 hours, auto-saves every stage)
```

**That's it!** Training will run automatically through all 5 stages.
Check back in 10-12 hours and download the backups from Output tab.